In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/tox21.csv.gz"
tox = pd.read_csv(url)
print("Tox21 geladen:", tox.shape)

Tox21 geladen: (8014, 14)


In [2]:
# Nur Substanzen mit SR-MMP-Label behalten
mmp = tox[tox["SR-MMP"].notna()].copy()
print("mit SR-MMP-Label:", len(mmp))

mit SR-MMP-Label: 5920


In [3]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

groesster_baustein = rdMolStandardize.LargestFragmentChooser()

def smiles_zu_key(smiles):
    mol = Chem.MolFromSmiles(smiles)   # Text -> Molekül (oder None)
    if mol is None:
        return None
    mol = groesster_baustein.choose(mol)   # Salz entfernen: größtes Teilchen
    return Chem.MolToInchiKey(mol)

In [4]:
# Keys für alle Substanzen berechnen
mmp["key"] = mmp["smiles"].apply(smiles_zu_key)
print("ohne lesbaren Key:", mmp["key"].isna().sum())

[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragmentChooser
[16:04:22] Running LargestFragment

ohne lesbaren Key: 6


[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Running LargestFragmentChooser
[16:04:27] Fragment: C1CNCCN1
[16:04:27] New largest fragment: C1CNCCN1 (16)

In [5]:
# JUMP laden und abgleichen
jump = pd.read_csv("https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/compound.csv.gz")
jump_voll   = set(jump["Metadata_InChIKey"].dropna())
jump_block1 = set(k.split("-")[0] for k in jump_voll)

keys = mmp["key"].dropna()
print("Substanzen mit Key:", len(keys))
print("Treffer ganzer Key:", keys.isin(jump_voll).sum())
print("Treffer erster Block:", keys.apply(lambda k: k.split("-")[0] in jump_block1).sum())

Substanzen mit Key: 5914
Treffer ganzer Key: 1087
Treffer erster Block: 1685


In [6]:
# Label-Verteilung unter den Treffern (erster Block)
def ist_treffer(k):
    if not isinstance(k, str):   # NaN/None (float) -> kein Treffer
        return False
    return k.split("-")[0] in jump_block1

treffer = mmp["key"].apply(ist_treffer)
print(mmp.loc[treffer, "SR-MMP"].value_counts())

SR-MMP
0.0    1436
1.0     249
Name: count, dtype: int64
